In [0]:
%run "../_shared/02_football_data_org_client"

In [0]:
import logging
import os

from typing import Any


logging.basicConfig(
    level=logging.INFO,
    format=(
        "%(asctime)s | "
        "%(levelname)s | "
        "%(message)s"
    ),
)

logger = logging.getLogger(__name__)

In [0]:
TOP_5_COMPETITIONS = {
    "PL": "premier_league",
    "PD": "la_liga",
    "BL1": "bundesliga",
    "SA": "serie_a",
    "FL1": "ligue_1",
}


BASE_URL = "https://api.football-data.org/v4"

RAW_BASE_PATH = (
    "/Volumes/football_data_platform/raw/source_files/"
    "football_data_org/competitions"
)


API_TOKEN = dbutils.secrets.get(
    scope="football-api",
    key="api-token",
)

In [0]:

def extract_competitions(
    client: FootballDataOrgClient,
    competitions: dict[str, str],
    output_directory: str,
) -> dict[str, int]:
    """
    Extract configured competitions from football-data.org.

    Each competition is stored as one raw JSON file containing
    the complete source API response.

    Existing files are skipped to keep reruns idempotent.
    """

    extracted_count = 0
    skipped_count = 0
    failed_count = 0

    for competition_code, competition_name in competitions.items():

        file_name = (
            f"{competition_code.lower()}_"
            f"{competition_name}.json"
        )

        file_path = os.path.join(
            output_directory,
            file_name,
        )

        if os.path.exists(file_path):
            logger.info(
                "Skipping existing competition: "
                "%s (%s)",
                competition_name,
                competition_code,
            )

            skipped_count += 1
            continue

        logger.info(
            "Extracting competition: %s (%s)",
            competition_name,
            competition_code,
        )

        try:
            payload = client.get_competition(
                competition_code=competition_code,
            )

            returned_code = payload.get("code")

            if returned_code != competition_code:
                raise FootballDataOrgError(
                    "Unexpected competition returned by API. "
                    f"Requested '{competition_code}', "
                    f"received '{returned_code}'."
                )

            write_json_atomic(
                data=payload,
                file_path=file_path,
            )

            extracted_count += 1

            logger.info(
                "Saved competition: %s -> %s",
                competition_code,
                file_path,
            )

        except FootballDataOrgRateLimitError:
            logger.exception(
                "Rate limit reached while extracting %s. "
                "Stopping ingestion.",
                competition_code,
            )

            raise

        except FootballDataOrgError:
            failed_count += 1

            logger.exception(
                "Failed to extract competition: %s",
                competition_code,
            )

        except Exception:
            failed_count += 1

            logger.exception(
                "Unexpected error while extracting competition: %s",
                competition_code,
            )

    return {
        "extracted": extracted_count,
        "skipped": skipped_count,
        "failed": failed_count,
    }

In [0]:

with FootballDataOrgClient(
    api_token=API_TOKEN,
    base_url=BASE_URL,
    min_request_interval_seconds=6.5,
    timeout_seconds=30,
) as client:

    ingestion_summary = extract_competitions(
        client=client,
        competitions=TOP_5_COMPETITIONS,
        output_directory=RAW_BASE_PATH,
    )

In [0]:
logger.info(
    "Competition ingestion completed. "
    "Extracted: %s | Skipped: %s | Failed: %s",
    ingestion_summary["extracted"],
    ingestion_summary["skipped"],
    ingestion_summary["failed"],
)

if ingestion_summary["failed"] > 0:
    raise RuntimeError(
        "Competition ingestion completed with failures. "
        f"Failed competitions: {ingestion_summary['failed']}"
    )

ingestion_summary